In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display

INVENTORY_DIR = Path('../output/inventory').resolve()

def load_csv(name):
    return pd.read_csv(INVENTORY_DIR / name, dtype=str,
                       keep_default_na=False, encoding='utf-8-sig')

docs = load_csv('documents.csv')
obs  = load_csv('metadata_observations.csv')
terms = load_csv('document_terms.csv')

# numeric helpers
for c in ['file_size_bytes', 'page_count', 'extracted_char_count', 'pages_without_text']:
    docs[c] = pd.to_numeric(docs[c], errors='coerce')

docs['has_review'] = docs['metadata_review_status'].replace('', 'unreviewed') != 'unreviewed'
docs['chars_per_page'] = docs['extracted_char_count'] / docs['page_count'].replace(0, np.nan)
docs['bytes_per_page'] = docs['file_size_bytes'] / docs['page_count'].replace(0, np.nan)
print(f'{len(docs)} docs, {len(obs)} obs, {len(terms)} terms')

93 docs, 311 obs, 116 terms


### Document Statistical analysis

In [7]:
docs.groupby('text_status')['page_count'].describe()

,count,mean,std,min,25%,50%,75%,max
text_status,,,,,,,,
extractable,85.0,33.682353,31.988172,1.0,14.0,24.0,38.0,159.0
no_text,1.0,1.000000,NaN,1.0,1.0,1.0,1.0,1.0
partial,7.0,45.857143,41.534839,17.0,22.0,23.0,63.5,110.0


In the above table we can see, that there are 85 documents with extractable status, one with no text and 7 with partial text. Lets focus on the 85 extractable documents. For that group the mean page count is ~34 pages, which is respectable size for a research papers. The distribution is right skewed with long tail on the left side. We can also notice that the standar deviation is big, which indicates that the points are spread out widely from the average value. We need also to consider the computational power during fine-tuning, if we have in mind the documents who are above the 75 percentile because of their size. This 

In [15]:
docs['characters_by_page'] = docs['extracted_char_count'] / docs['page_count']
docs.groupby('text_status')['characters_by_page'].describe().round(1)

,count,mean,std,min,25%,50%,75%,max
text_status,,,,,,,,
extractable,85.0,2934.7,1839.3,316.6,1637.6,2316.9,4107.3,6987.0
no_text,1.0,0.0,NaN,0.0,0.0,0.0,0.0,0.0
partial,7.0,927.7,429.7,511.3,606.9,672.2,1256.6,1583.3


In the above table we can see that character by page count across the collection averages 2,935 characters per page among fully extractable documents, with a median of 2,317 and an interquartile range of 1,638–4,107. The distribution is right-skewed, indicating a long upper tail of unusually dense documents rather than a symmetric spread. Seven partially extractable documents average 928 characters per page (median 672), roughly a third of the extractable group, consistent with pages that carry images, figures, or scanned inserts. A single document yielded no extractable text. Overall, the corpus is text-rich and well within normal ranges for born-digital PDFs, with only isolated anomalies at the tails.

In [23]:
def yr(s): return pd.to_numeric(s.astype(str).str.extract(r'(\d{4})')[0], errors='coerce')

created = (obs[obs.field_name=='pdf_creation_date']
           .assign(y=lambda d: yr(d['observed_value']))
           .groupby('document_id')['y'].min())
pub = docs.set_index('document_id')['year']
cmp = pd.concat([created.rename('created'), pub.rename('pub')], axis=1).dropna()
cmp['gap'] = cmp['created'] - cmp['pub']
display(cmp['gap'].describe())
print(f"Created before publication (impossible): {(cmp.gap < 0).sum()}")
print(f"Created >2y after publication (unreliable): {(cmp.gap > 2).sum()}")

count    0.0
mean     NaN
std      NaN
min      NaN
25%      NaN
50%      NaN
75%      NaN
max      NaN
Name: gap, dtype: float64

Created before publication (impossible): 0
Created >2y after publication (unreliable): 0


In [25]:
obs_with_status = obs.merge(docs[['document_id','text_status']], on='document_id')
cov = obs_with_status.groupby(['text_status','field_name'])['document_id'].nunique().unstack(fill_value=0)
sizes = docs['text_status'].value_counts()
cov_pct = (cov.T / sizes).T.round(3)   # coverage rate within each status group

print('Absolute field coverage per status:')
display(cov)
print('\nCoverage rate (0–1) within each status:')
display(cov_pct)

Absolute field coverage per status:


field_name,author,doi,online_date,pdf_creation_date,pdf_keywords,subject,title
text_status,,,,,,,
extractable,55,33,11,85,21,26,57
no_text,0,0,0,1,0,0,1
partial,7,0,0,7,0,0,7



Coverage rate (0–1) within each status:


field_name,author,doi,online_date,pdf_creation_date,pdf_keywords,subject,title
text_status,,,,,,,
extractable,0.647,0.388,0.129,1.0,0.247,0.306,0.671
no_text,0.000,0.000,0.000,1.0,0.000,0.000,1.000
partial,1.000,0.000,0.000,1.0,0.000,0.000,1.000


1. Text-derived fields vanish on partials
DOI, online_date, pdf_keywords all drop to 0% on partial and no_text. That's not a surprise: these are extracted from page text, and partial docs have pages without text. This is a structural limitation, not a quality problem — the text just isn't there to regex.

2. Embedded fields are better covered on partials
TItle and author are at 100% on partials vs 67% on extractables. That's the opposite of what I'd assume. This group is msotly homogeneous type of document with the same structure. This made it easy for the collection_inventory.ipynb to extract those fields.